# 06 Transformer Model - PubMedBERT

This notebook fine-tunes **PubMedBERT** on the combined **FakeHealth + HealthFact** binary misinformation dataset, using the **exact same** split strategy, hyperparameters, and evaluation pipeline as `04_transformer_biobert.ipynb` so that results are directly comparable.


## Why PubMedBERT?

Unlike BioBERT — which was initialised from general BERT and then continued pre-training on biomedical text — **PubMedBERT** (`microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext`) was pre-trained **from scratch** exclusively on PubMed abstracts and full-text articles. This means its vocabulary and representations are fully specialised to biomedical and clinical language, with no contamination from general-domain text.

Key differences vs BioBERT:

| Property | BioBERT | PubMedBERT |
|---|---|---|
| Initialisation | General BERT | Scratch |
| Pre-training data | PubMed + PMC | PubMed abstracts + full-text |
| Casing | Cased | **Uncased** |
| Vocabulary | General BERT vocab | Domain-specific vocab |

Because PubMedBERT is **uncased**, the tokenizer will lowercase all input text automatically — no manual pre-processing change is needed.


## GPU Training Note

This notebook is configured for **GPU training**. Key settings:

- `num_train_epochs = 3`
- `per_device_train_batch_size = 16`
- `gradient_accumulation_steps = 2`
- `max_length = 256`

If no GPU is detected at runtime, PyTorch will fall back to CPU automatically (training will be significantly slower). To force CPU, add `use_cpu=True` to `TrainingArguments`.


## Model Download
- `microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext`


In [ ]:
from pathlib import Path
import json
import os
import random

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

In [ ]:
PROJECT_ROOT = Path(r'C:\Users\ribam\Desktop\Reseach\Dataset')
DATA_PATH = PROJECT_ROOT / 'dataset' / 'processed' / 'fakehealth_healthfact_binary_clean.csv'
MODEL_DIR = PROJECT_ROOT / 'model' / 'pubmedbert_fakehealth_healthfact'
OUTPUT_DIR = PROJECT_ROOT / 'output' / 'pubmedbert_fakehealth_healthfact'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = 'microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext'
MAX_LENGTH = 256
NUM_EPOCHS = 3
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = 2
LEARNING_RATE = 2e-5
RANDOM_STATE = 42

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
print('Model:', MODEL_NAME)

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_STATE)

In [ ]:
df = pd.read_csv(DATA_PATH)
print('Shape:', df.shape)
print('Dataset counts:', df['dataset'].value_counts().to_dict())
print('Label counts:', df['label'].value_counts().to_dict())
df[['dataset', 'split', 'record_id', 'label', 'text']].head(3)

## Split Strategy

Identical to `04_transformer_biobert.ipynb` so results are directly comparable:

- Keep the original `healthfact` `train / dev / test` splits as-is.
- Create a stratified 70 / 15 / 15 split for `FakeHealth` (which has no predefined split).
- Merge the corresponding partitions.

`RANDOM_STATE = 42` is locked in both notebooks, so the exact same rows land in each split.


In [ ]:
healthfact_df = df[df['dataset'] == 'healthfact'].copy()
fakehealth_df = df[df['dataset'] == 'fakehealth'].copy()

fake_train, fake_temp = train_test_split(
    fakehealth_df,
    test_size=0.30,
    stratify=fakehealth_df['label'],
    random_state=RANDOM_STATE,
)
fake_dev, fake_test = train_test_split(
    fake_temp,
    test_size=0.50,
    stratify=fake_temp['label'],
    random_state=RANDOM_STATE,
)

train_df = pd.concat([healthfact_df[healthfact_df['split'] == 'train'], fake_train], ignore_index=True)
dev_df   = pd.concat([healthfact_df[healthfact_df['split'] == 'dev'],   fake_dev],   ignore_index=True)
test_df  = pd.concat([healthfact_df[healthfact_df['split'] == 'test'],  fake_test],  ignore_index=True)

print('Train shape:', train_df.shape)
print('Dev shape:  ', dev_df.shape)
print('Test shape: ', test_df.shape)
print('Train labels:', train_df['label'].value_counts().to_dict())
print('Dev labels:  ', dev_df['label'].value_counts().to_dict())
print('Test labels: ', test_df['label'].value_counts().to_dict())

In [ ]:
train_df.to_csv(OUTPUT_DIR / 'train_split.csv', index=False)
dev_df.to_csv(OUTPUT_DIR / 'dev_split.csv', index=False)
test_df.to_csv(OUTPUT_DIR / 'test_split.csv', index=False)
print('Saved train/dev/test split files to', OUTPUT_DIR)

## Load Tokenizer and Model

PubMedBERT uses an **uncased** tokenizer — the tokenizer itself handles lowercasing, so no manual text pre-processing change is needed compared to BioBERT.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: 'misinformation', 1: 'reliable'},
    label2id={'misinformation': 0, 'reliable': 1},
)

## Dataset Class and Data Collator


In [ ]:
class HealthDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length,
        )
        encoded['labels'] = int(self.labels[idx])
        return encoded

In [ ]:
train_dataset = HealthDataset(train_df['text'], train_df['label'], tokenizer, MAX_LENGTH)
dev_dataset   = HealthDataset(dev_df['text'],   dev_df['label'],   tokenizer, MAX_LENGTH)
test_dataset  = HealthDataset(test_df['text'],  test_df['label'],  tokenizer, MAX_LENGTH)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

len(train_dataset), len(dev_dataset), len(test_dataset)

## Metrics Function


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average='binary',
        zero_division=0,
    )
    accuracy = accuracy_score(labels, predictions)
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
    }

## Training Arguments

All hyperparameters are identical to BioBERT to ensure a fair comparison.


In [ ]:
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / 'checkpoints'),
    do_train=True,
    do_eval=True,
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_strategy='steps',
    logging_steps=50,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=0.01,
    warmup_ratio=0.1,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    save_total_limit=2,
    report_to='none',
    seed=RANDOM_STATE,
    remove_unused_columns=True,
    group_by_length=True,
)

## Train


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

## Evaluate on Dev and Test Sets


In [ ]:
dev_metrics  = trainer.evaluate(dev_dataset,  metric_key_prefix='eval')
test_metrics = trainer.evaluate(test_dataset, metric_key_prefix='eval')

print('--- Dev Metrics ---')
for k, v in dev_metrics.items():
    print(f'  {k}: {v}')

print('\n--- Test Metrics ---')
for k, v in test_metrics.items():
    print(f'  {k}: {v}')

## Test Set Classification Report


In [ ]:
test_predictions  = trainer.predict(test_dataset)
test_pred_labels  = np.argmax(test_predictions.predictions, axis=-1)
test_probabilities = torch.softmax(torch.tensor(test_predictions.predictions), dim=1).numpy()

print(classification_report(test_df['label'], test_pred_labels, digits=4))

## Confusion Matrix


In [ ]:
os.environ['MPLCONFIGDIR'] = str(OUTPUT_DIR / '.matplotlib')
(OUTPUT_DIR / '.matplotlib').mkdir(parents=True, exist_ok=True)
import matplotlib.pyplot as plt

cm = confusion_matrix(test_df['label'], test_pred_labels)
fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(cm, cmap='Blues')
ax.set_title('PubMedBERT Test Confusion Matrix')
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, cm[i, j], ha='center', va='center', color='black')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'pubmedbert_confusion_matrix.png', dpi=150)
plt.show()

## Save Model, Metrics, and Predictions


In [ ]:
trainer.save_model(str(MODEL_DIR))
tokenizer.save_pretrained(str(MODEL_DIR))

metrics = {
    'model_name': MODEL_NAME,
    'max_length': MAX_LENGTH,
    'num_epochs': NUM_EPOCHS,
    'train_batch_size': TRAIN_BATCH_SIZE,
    'eval_batch_size': EVAL_BATCH_SIZE,
    'gradient_accumulation_steps': GRADIENT_ACCUMULATION_STEPS,
    'learning_rate': LEARNING_RATE,
    'device': DEVICE,
    'train_rows': int(len(train_df)),
    'dev_rows': int(len(dev_df)),
    'test_rows': int(len(test_df)),
    'dev_metrics':  {k: float(v) for k, v in dev_metrics.items()  if isinstance(v, (int, float))},
    'test_metrics': {k: float(v) for k, v in test_metrics.items() if isinstance(v, (int, float))},
}

(OUTPUT_DIR / 'pubmedbert_metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')

prediction_df = test_df[['dataset', 'split', 'record_id', 'label', 'text']].copy()
prediction_df['prediction']    = test_pred_labels
prediction_df['prob_class_0']  = test_probabilities[:, 0]
prediction_df['prob_class_1']  = test_probabilities[:, 1]
prediction_df.to_csv(OUTPUT_DIR / 'pubmedbert_test_predictions.csv', index=False)

print('Saved model to:', MODEL_DIR)
print('Saved outputs to:', OUTPUT_DIR)
metrics

## Next Step

With PubMedBERT results saved, the next notebook is:

- **`07_model_comparison_three_way.ipynb`** — extend `05_model_comparison_and_error_analysis` to include PubMedBERT alongside TF-IDF + Logistic Regression and BioBERT for a three-way comparison table, bar chart, per-dataset breakdown, and updated error-analysis buckets.
